### 分块读取工具函数

In [2]:
import os

from typing import BinaryIO

def find_chunk_boudaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    把文件切分成可以独立计数的块（chunk），
    若边界重叠（文件太小或者分隔符太稀疏），返回块的数量会少于预期
    """
    # 确保传入的是字节串类型，因为文件以二进制模式读取
    assert isinstance(split_special_token, bytes), "必须使用字节串（bytes）表示特殊的 Token"

    # 获取文件总字节大小
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)  # 重置文件回到开头

    # 计算初步的理想的块大小（字节数）
    chunk_size = file_size // desired_num_chunks

    # 初始的边界猜测，根据块大小进行均匀分布
    # 边界的数组包括起始位置0和结束的位置file_size
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size # 确保最后一个边界指向文件末尾

    mini_chunk_size = 4096 # 每次向后搜索缓冲区大小4k字节

    # 遍历除了开头和结尾之外的所有中间边界点
    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position) # 跳转到初步猜测的边界位置

        while True:
            # 读取一小块数据进行扫描
            mini_chunk = file.read(mini_chunk_size)

            # 若读取到文件末尾（EOF），说明后面没有分隔符，设为文件末尾
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # 在读取的块中查找分隔符
            found_at = mini_chunk.find(split_special_token)

            if found_at != -1:
                # 找到分隔符在将边界调整到分隔符的确定位置
                chunk_boundaries[bi] = initial_position + found_at
                break

            # 没找到继续向后移动指针进行下一轮搜索
            initial_position += mini_chunk_size

    # 使用set()去除重复的边界，防止多个猜测点指向同一个分隔符
    # sorted()确保边界按从小到大的顺序排列
    return sorted(set(chunk_boundaries))

### 按块读取文本

In [3]:
import time
from tqdm import tqdm

def iter_text_chunks_with_monitor(
    file_path: str,
    chunk_size: int = 1_000_000, # 1MB
    log_every: int = 5  # 每隔多少个chunk记录一次日志
):
    start_time = time.time()
    bytes_processed = 0
    chunk_count = 0

    with open(file_path, "r", encoding="utf-8") as f:
        buffer = []
        buffer_size = 0

        for line in f:
            buffer.append(line)
            buffer_size += len(line)
            bytes_processed += len(line)

            if buffer_size >= chunk_size:
                yield "".join(buffer)
                buffer = []
                buffer_size = 0
                chunk_count += 1

                if chunk_count % log_every == 0:
                    log_status(
                        prefix = "分词器流式处理",
                        bytes_processed=bytes_processed,
                        start_time=start_time,
                    )
        if buffer:
            yield "".join(buffer)

### 训练BPE分词器

#### step1:在不修改tokenizer内部实现的前提下，实时监控内存占用和数据吞吐量，观察tokenizer训练过程中的系统行为

In [4]:
# 当前进程所占的内存
import psutil

def get_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024


In [5]:
# 日志状态函数输出（内存占用、处理数据量、处理速度）
def log_status(prefix, bytes_processed, start_time):
    elapsed = time.time() - start_time
    mb = bytes_processed / 1024 / 1024
    throughput = mb / elapsed if elapsed > 0 else 0.0 # 计算吞吐量即每秒处理的数据量
    mem = get_memory_mb()

    print(
        f"{prefix} |"
        f"mem={mem:7.1f}MB |"
        f"data={mb:8.1f}MB |"
        f"speed={throughput:6.2f}MB/s"
    )

#### 训练BPE Tokenizer

In [8]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFKC
from typing import Union

def train_bpe_tokenizer(
    train_file: str,
    val_file: Union[str, None] = None,
    vocab_size: int = 50257,
    num_chunks: int = 8,
    output_dir: str = "./bpr_tokenizer",
):
    os.makedirs(output_dir, exist_ok=True)

    special_tokens = [
        "<|endoftext|>",
        "<|unk|>",
        "<|pad|>",
        "<|mask|>",
        "<|eos|>",
    ]

    tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))
    tokenizer.normalizer = NFKC()

    # 设计GPT-2风格的BPE Tokenizer
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        show_progress=True,
    )

    def text_iterator():
        # 训练集
        for chunk in iter_text_chunks_with_monitor(
            train_file,
            chunk_size=1_000_000, # 每次处理1MB数据，处理完就会清除数据方便继续处理
            log_every=20,
        ):
            yield chunk
        
        # 验证集
        if val_file is not None:
            for chunk in iter_text_chunks_with_monitor(
                val_file,
                chunk_size=1_000_000,
                log_every=10,
            ):
                yield chunk
    
    print("开始训练BPE Tokenizer...")
    tokenizer.train_from_iterator(text_iterator(), trainer=trainer)
    print("训练完成，正在保存Tokenizer...")
    tokenizer.save(os.path.join(output_dir, "tokenizer.json"))
    print(f"分词器已保存到{output_dir}/tokenizer.json")

    return tokenizer

if __name__ == "__main__":
    train_path = "./TinyStoriesV2-GPT4-train.txt"
    val_path = "./TinyStoriesV2-GPT4-valid.txt"
    tokenizer = train_bpe_tokenizer(
        train_file=train_path,
        val_file=val_path,
        vocab_size=50257, # 词表大小一般取32000或50257等
        num_chunks=16, # 读取文件的分块数量（根据内存大小调整）
        output_dir="./bpe_tokenizer",
    )
            

开始训练BPE Tokenizer...
分词器流式处理 |mem=  108.4MB |data=    19.1MB |speed=222.43MB/s
分词器流式处理 |mem=  127.4MB |data=    38.2MB |speed=228.93MB/s
分词器流式处理 |mem=  146.4MB |data=    57.2MB |speed=215.31MB/s
分词器流式处理 |mem=  165.5MB |data=    76.3MB |speed=207.74MB/s
分词器流式处理 |mem=  184.6MB |data=    95.4MB |speed=214.62MB/s
分词器流式处理 |mem=  203.7MB |data=   114.5MB |speed=217.27MB/s
分词器流式处理 |mem=  222.8MB |data=   133.5MB |speed=222.49MB/s
分词器流式处理 |mem=  242.0MB |data=   152.6MB |speed=225.95MB/s
分词器流式处理 |mem=  261.1MB |data=   171.7MB |speed=226.13MB/s
分词器流式处理 |mem=  280.2MB |data=   190.8MB |speed=227.42MB/s
分词器流式处理 |mem=  299.4MB |data=   209.8MB |speed=229.28MB/s
分词器流式处理 |mem=  318.5MB |data=   228.9MB |speed=231.04MB/s
分词器流式处理 |mem= 1744.6MB |data=   248.0MB |speed= 19.69MB/s
分词器流式处理 |mem= 1563.4MB |data=   267.1MB |speed= 20.77MB/s
分词器流式处理 |mem=  153.4MB |data=   286.1MB |speed= 21.01MB/s
分词器流式处理 |mem=  172.5MB |data=   305.2MB |speed= 22.30MB/s
分词器流式处理 |mem=  190.6MB |data=   324.3MB |speed= 23.

#### 验证训练的BPE Tokenizer

In [ ]:
encoded = tokenizer.encode(" Hello, world! <|endoftext|>")
print(encoded.tokens)
print(encoded.ids)
print(tokenizer.decode([1501])) # 输出前带一个空格的" world"

# 将G替换回空格
clean_tokens = [t.replace("Ġ", " ") for t in encoded.tokens]
print(clean_tokens)

['ĠHello', ',', 'Ġworld', '!', 'Ġ', '<|endoftext|>']
[8431, 16, 1501, 5, 149, 0]
 world
[' Hello', ',', ' world', '!', ' ', '<|endoftext|>']


GPT-2正则化风格训练的tokenizer在单词前添加一个Ġ表示带有空格，因为对模型来说带空格和不带空格的Token不同

In [10]:
# Token统计函数
def analyze_tokenizer(tokenizer, texts):
    lengths = [len(tokenizer.encode(t).ids) for t in texts]
    return {
        "avg_token": sum(lengths) / len(lengths) if lengths else 0,
        "max_token": max(lengths) if lengths else 0,
    }

In [11]:
import random

def load_stories(file_path, num_samples=None):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    stories = [s.strip() for s in content.split("<|endoftext|>") if s.strip()]

    if num_samples is not None:
        n = min(num_samples, len(stories))
        stories = random.sample(stories, k=n)
    return stories

train_path = "./TinyStoriesV2-GPT4-train.txt"
val_path = "./TinyStoriesV2-GPT4-valid.txt"
test_texts = load_stories(val_path, num_samples=10) # 从验证集中随机抽取10条故事进行分析
train_texts = load_stories(train_path, num_samples=20) # 从训练集中随机抽取20条故事进行分析
print(f"随机抽取的训练样本数：{len(train_texts)}")
print(f"随机抽取的测试样本数：{len(test_texts)}")

# 分析训练集和验证集的token统计
train_stats = analyze_tokenizer(tokenizer, train_texts)
val_stats = analyze_tokenizer(tokenizer, test_texts)
print("训练集Token统计：", train_stats)
print("验证集Token统计：", val_stats)

随机抽取的训练样本数：20
随机抽取的测试样本数：10
训练集Token统计： {'avg_token': 209.9, 'max_token': 592}
验证集Token统计： {'avg_token': 163.6, 'max_token': 211}
